# Explore llm-d-inference-sim

**Baseline:** Flags → Start (fake-model) → I/O  
**KV cache:** Flags KV → Start render → Start sim → I/O (ugyanaz a prompt 2×) → Stop KV

Baseline port **18001**. KV render host **18082**, KV sim host **18001** (ne futtasd egyszerre a baseline Starttal).

## 1. Flags (baseline)

In [ ]:
IMAGE = "ghcr.io/llm-d/llm-d-inference-sim:v0.9.0"
HOST_PORT = 18001
CONTAINER_NAME = "explore-sim"
SIM_URL = f"http://127.0.0.1:{HOST_PORT}"

MODEL = "fake-model"
MODE = "random"  # or "echo"

FLAGS = [
    "--model", MODEL,
    "--port", "8000",
    "--mode", MODE,
    "--max-model-len", "4096",
    "--time-to-first-token", "100ms",
    "--inter-token-latency", "20ms",
    "--max-num-seqs", "4",
    "--max-waiting-queue-length", "1000",
    "--failure-injection-rate", "0",
]

print(" ".join(FLAGS))
print(SIM_URL)

## 2. Start (baseline)

Remove old container, run new one, wait for `/health`.

In [ ]:
import subprocess
import time
from urllib.request import urlopen

subprocess.run(["docker", "rm", "-f", CONTAINER_NAME], check=False)
subprocess.run(
    ["docker", "run", "-d", "--rm", "--name", CONTAINER_NAME,
     "-p", f"{HOST_PORT}:8000", IMAGE, *FLAGS],
    check=True,
)

for _ in range(30):
    try:
        status = urlopen(f"{SIM_URL}/health", timeout=2).status
        print("health", status, SIM_URL)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError(f"health timeout: {SIM_URL}")

## 3. Input + output (baseline)

In [ ]:
import json
from urllib.request import Request, urlopen

request_body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "EchoProbe-42: the quick brown fox"}],
    "max_tokens": 32,
    "stream": False,
}
print("=== INPUT ===")
print(json.dumps(request_body, indent=2, ensure_ascii=False))

req = Request(
    f"{SIM_URL}/v1/chat/completions",
    data=json.dumps(request_body).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urlopen(req, timeout=60) as resp:
    response = json.loads(resp.read().decode())

print("\n=== OUTPUT (chat.completion) ===")
print(json.dumps(response, indent=2, ensure_ascii=False))

## Stop baseline (optional)

In [ ]:
subprocess.run(["docker", "rm", "-f", CONTAINER_NAME], check=False)
print("stopped", CONTAINER_NAME)

---

## 4. Flags KV

Cél: `enable-kvcache` + Qwen + render → ugyanaz a prompt **kétszer**; a 2. válasznál `cached_tokens` > 0.
(`kv_transfer_params` lehet `null` — az P/D, nem a helyi cache.)

In [2]:
IMAGE = "ghcr.io/llm-d/llm-d-inference-sim:v0.9.0"
RENDER_IMAGE = "vllm/vllm-openai-cpu:v0.21.0"
NETWORK = "explore-kv-net"

RENDER_NAME = "explore-render"
RENDER_HOST_PORT = 18082
RENDER_URL_IN_DOCKER = f"http://{RENDER_NAME}:8082"
RENDER_URL_HOST = f"http://127.0.0.1:{RENDER_HOST_PORT}"

CONTAINER_NAME = "explore-sim-kv"
HOST_PORT = 18001
SIM_URL = f"http://127.0.0.1:{HOST_PORT}"
POD_IP = "127.0.0.1"

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODE = "random"

FLAGS = [
    "--model", MODEL,
    "--port", "8000",
    "--mode", MODE,
    "--max-model-len", "4096",
    "--time-to-first-token", "100ms",
    "--inter-token-latency", "20ms",
    "--max-num-seqs", "4",
    "--max-waiting-queue-length", "1000",
    "--enable-kvcache",
    "--render-url", RENDER_URL_IN_DOCKER,
]

print("MODEL", MODEL)
print("FLAGS", " ".join(FLAGS))
print("SIM_URL", SIM_URL)
print("RENDER_URL_HOST", RENDER_URL_HOST)
print("POD_IP", POD_IP)

MODEL Qwen/Qwen2.5-1.5B-Instruct
FLAGS --model Qwen/Qwen2.5-1.5B-Instruct --port 8000 --mode random --max-model-len 4096 --time-to-first-token 100ms --inter-token-latency 20ms --max-num-seqs 4 --max-waiting-queue-length 1000 --enable-kvcache --render-url http://explore-render:8082
SIM_URL http://127.0.0.1:18001
RENDER_URL_HOST http://127.0.0.1:18082
POD_IP 127.0.0.1


## 5. Start render

Első futásnál az image / modell letöltés sokáig tarthat.

In [3]:
import subprocess
import time
from urllib.request import urlopen

subprocess.run(["docker", "network", "create", NETWORK], check=False)
subprocess.run(["docker", "rm", "-f", RENDER_NAME], check=False)

subprocess.run(
    [
        "docker", "run", "-d", "--rm",
        "--name", RENDER_NAME,
        "--network", NETWORK,
        "-p", f"{RENDER_HOST_PORT}:8082",
        "--entrypoint", "vllm",
        RENDER_IMAGE,
        "launch", "render", MODEL, "--port=8082",
    ],
    check=True,
)

for i in range(180):
    try:
        status = urlopen(f"{RENDER_URL_HOST}/health", timeout=3).status
        print("render health", status, RENDER_URL_HOST)
        break
    except Exception as exc:
        if i % 10 == 0:
            print(f"waiting render... ({i}s) {exc}")
        time.sleep(1)
else:
    raise RuntimeError(f"render health timeout: {RENDER_URL_HOST}")

waiting render... (0s) Remote end closed connection without response
waiting render... (10s) Remote end closed connection without response
waiting render... (20s) Remote end closed connection without response
waiting render... (30s) Remote end closed connection without response
waiting render... (40s) Remote end closed connection without response
waiting render... (50s) [WinError 10054] A létező kapcsolatot a távoli állomás kényszerítetten bezárta
waiting render... (60s) Remote end closed connection without response
waiting render... (70s) Remote end closed connection without response
render health 200 http://127.0.0.1:18082


## 6. Start sim (KV)

In [4]:
subprocess.run(["docker", "rm", "-f", CONTAINER_NAME], check=False)

subprocess.run(
    [
        "docker", "run", "-d", "--rm",
        "--name", CONTAINER_NAME,
        "--network", NETWORK,
        "-e", f"POD_IP={POD_IP}",
        "-p", f"{HOST_PORT}:8000",
        IMAGE,
        *FLAGS,
    ],
    check=True,
)

for _ in range(60):
    try:
        status = urlopen(f"{SIM_URL}/health", timeout=2).status
        print("sim health", status, SIM_URL)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError(f"sim health timeout: {SIM_URL}")

sim health 200 http://127.0.0.1:18001


## 7. Input + output (KV) — ugyanaz a prompt kétszer

Nézd a `usage` → `cached_tokens` (vagy `prompt_tokens_detail` / `prompt_tokens_details`) értékét az 1. és 2. válasznál.

In [7]:
import json
from urllib.request import Request, urlopen


def cached_tokens(resp: dict) -> object:
    usage = resp.get("usage") or {}
    detail = (
        usage.get("prompt_tokens_details")
        or usage.get("prompt_tokens_detail")
        or {}
    )
    return detail.get("cached_tokens")


def chat_once(body: dict) -> dict:
    req = Request(
        f"{SIM_URL}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())


request_body1 = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "EchoProbe-42: the quick brown fox"}],
    "max_tokens": 32,
}

request_body2 = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "This is a second request, which is totally different."}],
    "max_tokens": 32,
}

print("=== INPUT ===")
print(json.dumps(request_body1, indent=2, ensure_ascii=False))
print(json.dumps(request_body2, indent=2, ensure_ascii=False))

r1 = chat_once(request_body1)
print("\n=== OUTPUT #1 ===")
print(json.dumps(r1, indent=2, ensure_ascii=False))
print("cached_tokens #1:", cached_tokens(r1))

r2 = chat_once(request_body2)
print("\n=== OUTPUT #2 (same prompt) ===")
print(json.dumps(r2, indent=2, ensure_ascii=False))
print("cached_tokens #2:", cached_tokens(r2))

=== INPUT ===
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "messages": [
    {
      "role": "user",
      "content": "EchoProbe-42: the quick brown fox"
    }
  ],
  "max_tokens": 32
}
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "messages": [
    {
      "role": "user",
      "content": "This is a second request, which is totally different."
    }
  ],
  "max_tokens": 32
}

=== OUTPUT #1 ===
{
  "id": "chatcmpl-ffc5cef5-73d0-502c-81b4-0de7f689b1ea",
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "object": "chat.completion",
  "kv_transfer_params": null,
  "created": 1791968617,
  "usage": {
    "prompt_tokens": 39,
    "completion_tokens": 5,
    "total_tokens": 44,
    "prompt_tokens_detail": {
      "cached_tokens": 32
    }
  },
  "choices": [
    {
      "index": 0,
      "finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "Today is a nice sunny "
      }
    }
  ]
}
cached_tokens #1: 32

=== OUTPUT #2 (same prompt) ===
{
  "id": "chatcmpl-f191

## Stop KV (render + sim)

In [ ]:
subprocess.run(["docker", "rm", "-f", CONTAINER_NAME], check=False)
subprocess.run(["docker", "rm", "-f", RENDER_NAME], check=False)
print("stopped", CONTAINER_NAME, "and", RENDER_NAME)